# 📧 Automação de E-mails — Projudi
Encaminha e-mails com PDF recebidos e registra respostas no sistema Projudi.

---

In [21]:
# ============================================================
# IMPORTS
# ============================================================
import smtplib
import imaplib
import email
from email.message import EmailMessage
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart
from email.mime.image import MIMEImage
from email.utils import make_msgid
from email.header import decode_header
from email.utils import parsedate_to_datetime
from imap_tools import MailBox, A
# from session_id import get_cookies

import json
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse
import pandas as pd
import re
from collections import defaultdict

import os

import csv
from itertools import zip_longest
import time
import random
from datetime import datetime
import pytz

from selenium import webdriver
from selenium.webdriver.firefox.options import Options
from selenium.webdriver.firefox.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException
from selenium.webdriver.support.ui import Select
from webdriver_manager.firefox import GeckoDriverManager
from selenium.webdriver.firefox.firefox_profile import FirefoxProfile

In [22]:
import sys
# print(sys.executable)
sys.path.append(r"E:\send_of")

from session_id import get_cookies  # seu arquivo session_id.py com a função
url = "https://projudi.tjba.jus.br/projudi/"
cookies = get_cookies(domain=url)
print(cookies)

# # import browser_cookie3
# from session_id import get_cookies

# url = "https://projudi.tjba.jus.br/projudi/"
# cookies = get_cookies(domain=url)
# print(cookies)

{'JSESSIONID': '8BBA4E8FFB4F1298520AC432CBA85954.tomcat09-01', 'ADC_CONN_539B3595F4E': '19AB97A6995D790C773C15EEB5AFCF6F8FB53AAAED6C261A4ACB5D23455C59630E9393BD07A581E7', 'ADC_REQ_2E94AF76E7': 'FC548B0AB68C9C8B7DC0E04A12A63D9F5460B5F1DE741793F56F3D2FA0BDCB0F218B795FCAA762CA'}


In [23]:
# ============================================================
# CONFIGURAÇÕES GERAIS
# ============================================================
# link_base = url = "https://projudi.tjba.jus.br/projudi/"
# cookies = get_cookies(domain=link_base)

tempo_espera      = random.uniform(1, 5)
senha_app         = 'ouysuorpvqprfqig'
usuario           = 'pafonso.2vsj@gmail.com'
IMAP_SERVER       = 'imap.gmail.com'
NUM_MAX_EMAILS    = 100
PADRAO_ASSUNTO    = re.compile(r"2.*VSJ", re.IGNORECASE)

SMTP_SERVER   = 'smtp.gmail.com'
SMTP_PORT     = 587
SMTP_USER     = usuario
SMTP_PASSWORD = senha_app
REMETENTE     = SMTP_USER
DESTINATARIO  = 'pafonso-2vsj@tjba.jus.br'

path_csv = 'protocolo_email_projudi.csv'

CAMPOS = [
    'processo', 'num_oficio', 'email_destino', 'status',
    'data_envio', 'hora_envio', 'registrar_envio', 'resposta', 'msg_id',
    'url_oficio', 'url_processo', 'url_recebimento', 'url_baixa', 'assunto',
]

# PDFs conhecidos que devem ser ignorados (lista negra)
PDFS_IGNORADOS = {
    '3.Sgt PM Kelson e Sgt PM Marilson 204.2025.sec_0001.pdf',
    'Oficio_00126551581.pdf',
    'Extrato_00122965493_2VSJ_SGT_SANTOS.pdf',
    'ADILSON MOREIRA_0001.pdf',
    'Oficio_00124493202.pdf',
    'PROCESSO_ 0000244-56.2025.8.17.3120 - CARTA PRECATÓRIA CÍVEL - 0000244-56.2025.8.17.3120-1762177111273-864398-processo.pdf',
    'Requerimento_0087935422_EMAIL_1.pdf',
    'Oficio_00122965336.pdf',
}

In [24]:
# ============================================================
# CONFIGURAÇÃO PROJUDI (cookies e URLs)
# ============================================================
link_base   = 'https://projudi.tjba.jus.br/projudi/'
url_oficios   = link_base + 'listagens/CumprimentoCartorio?tipo=oficio&acao=expedidos'

parsed        = urlparse(url_oficios)
cookie_domain = parsed.hostname

# cookies = cookies
# {
#     'ADC_CONN_539B3595F4E': '574CAA1662357EBBF5F94ED94FF66FE08C10454FD7424D566BCB2DD33BADE9781AE5CBFD4E50793B',
#     'ADC_REQ_2E94AF76E7':   '17EB80EF2601FF13E17C4D41F54A5E701355460BBCC88D1E3722D8C48FB521953AE37A4688C180D2',
#     'ADRUM':                's~1773170981656&r~aHR0cHMlM0ElMkYlMkZwcm9qdWRpLnRqYmEuanVzLmJyJTJGcHJvanVkaSUyRg==',
#     'JSESSIONID':           'B35B5F86AE9CED4135E5A05BE4E0EA6D.tomcat09-03',
# }

headers = {
    'User-Agent':      'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 '
                       '(KHTML, like Gecko) Chrome/115.0.0.0 Safari/537.36',
    'Accept-Language': 'pt-BR,pt;q=0.9,en;q=0.8',
    'Referer':         url_oficios,
    'Accept':          'text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8',
}

In [25]:
# ============================================================
# GERENCIAMENTO DO CSV
# ============================================================

def inicializar_csv(path_csv):
    """Cria o arquivo CSV se não existir."""
    if not os.path.exists(path_csv):
        print(f"[INFO] Criando novo arquivo: {path_csv}")
        with open(path_csv, 'w', encoding='utf-8', newline='') as f:
            writer = csv.DictWriter(f, fieldnames=CAMPOS, delimiter=';')
            writer.writeheader()


def carregar_csv(path_csv):
    """Carrega o CSV removendo duplicados. Retorna (dados, msg_ids_registrados)."""
    dados               = []
    msg_ids_registrados = set()
    chaves_vistas       = set()

    with open(path_csv, encoding='utf-8', newline='') as f:
        reader = csv.DictReader(f, delimiter=';')
        for row in reader:
            # Ignora linha de cabeçalho duplicada
            if str(row.get('processo', '')).strip().lower() == 'processo':
                continue

            # Normaliza campos
            for campo in CAMPOS:
                row[campo] = str(row.get(campo) or '').strip()

            # Chave composta para detectar duplicados
            chave  = f"{row['assunto'].lower()}_{row['data_envio']}_{row['hora_envio']}"
            msg_id = row.get('msg_id', '').strip()

            if chave in chaves_vistas or (msg_id and msg_id in msg_ids_registrados):
                continue

            chaves_vistas.add(chave)
            if msg_id:
                msg_ids_registrados.add(msg_id)

            dados.append(row)

    print(f"[INFO] CSV carregado: {len(dados)} registros, {len(msg_ids_registrados)} msg_ids únicos.")
    return dados, msg_ids_registrados


def salvar_csv(path_csv, dados):
    """Salva os dados no CSV."""
    with open(path_csv, 'w', encoding='utf-8', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=CAMPOS, delimiter=';')
        writer.writeheader()
        writer.writerows(dados)
    print(f"[✓] CSV salvo: {len(dados)} registros.")


def registrar_encaminhamento(msg, dados, msg_ids_registrados, path_csv):
    """Registra um e-mail encaminhado no CSV."""
    agora    = datetime.now()
    msg_id = get_message_id(msg)  # ✅ Permanente, com fallbacks
    assunto  = msg.subject or '(sem assunto)'

    nome_arquivo = ''
    for anexo in msg.attachments:
        nome = (anexo.filename or '').strip()
        if nome.lower().endswith('.pdf'):
            nome_arquivo = nome
            break

    nova_linha = {
        'processo':        '',
        'num_oficio':      '',
        'email_destino':   msg.from_,
        'status':          'Enviado',
        'data_envio':      agora.strftime('%d/%m/%Y'),
        'hora_envio':      agora.strftime('%H:%M:%S'),
        'registrar_envio': 'juntado',
        'resposta':        '',
        'msg_id':          msg_id,
        'url_oficio':      '',
        'url_processo':    '',
        'url_recebimento': '',
        'url_baixa':       '',
        'assunto':         assunto,
    }

    dados.append(nova_linha)
    msg_ids_registrados.add(msg_id)

    # Append direto no CSV para não perder dados se o processo cair
    with open(path_csv, 'a', encoding='utf-8', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=CAMPOS, delimiter=';')
        writer.writerow(nova_linha)

    print(f"📄 Registrado: {assunto} | msg_id: {msg_id}")


# Inicializa e carrega
inicializar_csv(path_csv)
dados, msg_ids_registrados = carregar_csv(path_csv)

[INFO] CSV carregado: 166 registros, 156 msg_ids únicos.


In [26]:
# ============================================================
# FUNÇÕES AUXILIARES
# ============================================================

def get_message_id(msg):
    """Extrai message_id permanente do e-mail, com fallbacks."""
    # 1. Atributo direto da lib imap_tools
    mid = getattr(msg, 'message_id', '') or ''
    if mid.strip():
        return mid.strip()

    # 2. Header interno do objeto MIME
    try:
        mid = msg.obj.get('Message-ID', '') or ''
        if mid.strip():
            return mid.strip()
    except Exception:
        pass

    # 3. Fallback: uid da sessão IMAP (menos confiável, mas evita ignorar o e-mail)
    uid = str(getattr(msg, 'uid', '') or '').strip()
    if uid:
        print(f"⚠️  Usando UID como fallback para: {msg.subject}")
        return f'uid:{uid}'

    return ''


def normalizar_assunto(assunto):
    """Remove prefixos Re/Enc/Fwd e normaliza para minúsculas."""
    assunto = assunto.lower()
    assunto = re.sub(r'^(re|res|enc|fwd|fw)\s*:\s*', '', assunto)
    return assunto.strip()


def limpar_cid(texto):
    """Remove referências de imagem inline [cid:...]."""
    return re.sub(r'\[cid:[^\]]+\]', '', texto)


def tem_pdf(msg):
    """Retorna True se houver PDF anexado fora da lista negra."""
    for anexo in msg.attachments:
        filename = (anexo.filename or '').strip()
        if filename in PDFS_IGNORADOS:
            continue
        if filename.lower().endswith('.pdf'):
            return True
    return False


def montar_assuntos_map(dados):
    """Monta dicionário de assunto normalizado → linha do CSV."""
    return {
        normalizar_assunto(row['assunto']): row
        for row in dados
        if row.get('assunto')
    }


def digitar(texto, campo):
    """Simula digitação humana no campo Selenium."""
    for char in texto:
        campo.send_keys(char)
        time.sleep(random.uniform(0.02, 0.08))


In [27]:
# ============================================================
# ENVIO DE E-MAIL
# ============================================================

def encaminhar_email_completo(msg):
    """Encaminha e-mail com PDFs para o destinatário configurado."""
    email_msg = EmailMessage()
    email_msg['Subject'] = f"Enc: {msg.subject or '(sem assunto)'}"
    email_msg['From']    = REMETENTE
    email_msg['To']      = DESTINATARIO

    corpo_original = msg.text or msg.html or '(sem conteúdo)'
    corpo = f"""
Encaminhado automaticamente.

────────────────────────────────────
📨 Remetente original: {msg.from_}
📅 Data: {msg.date.strftime('%d/%m/%Y %H:%M')}
📄 Assunto original: {msg.subject}
📎 Anexos: {[a.filename for a in msg.attachments if a.filename]}
────────────────────────────────────

{corpo_original}
"""
    email_msg.set_content(corpo)

    pdfs_anexados = 0
    for anexo in msg.attachments:
        nome = (anexo.filename or '').strip()
        if nome.lower().endswith('.pdf'):
            email_msg.add_attachment(
                anexo.payload, maintype='application', subtype='pdf', filename=nome
            )
            pdfs_anexados += 1

    if pdfs_anexados == 0:
        print(f"⚠️ Nenhum PDF para encaminhar: {msg.subject}")
        return False

    try:
        with smtplib.SMTP(SMTP_SERVER, SMTP_PORT) as smtp:
            smtp.starttls()
            smtp.login(SMTP_USER, SMTP_PASSWORD)
            smtp.send_message(email_msg)
        print(f"✅ Encaminhado: '{msg.subject}' → {DESTINATARIO}")
        return True
    except Exception as e:
        print(f"❌ Erro ao encaminhar '{msg.subject}': {e}")
        return False

In [28]:
# ============================================================
# JUNTADA NO PROJUDI (Selenium)
# ============================================================

def juntada_resposta_projudi(destinatario, url_recebimento, assunto, cookies,
                              cookie_domain, link_base,
                              digitar, tempo_espera, data_resposta, hora_resposta):
    profile = FirefoxProfile()
    profile.set_preference('general.useragent.override',
                           'Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:115.0) Gecko/20100101 Firefox/115.0')
    profile.set_preference('intl.accept_languages', 'pt-BR,pt;q=0.9,en;q=0.8')

    options = Options()
    options.profile = profile

    driver = webdriver.Firefox(
        service=Service(GeckoDriverManager().install()), options=options
    )
    driver.set_window_size(1280, 800)
    driver.get(link_base)
    time.sleep(tempo_espera)

    for name, value in cookies.items():
        driver.add_cookie({
            'name': name, 'value': value,
            'path': '/', 'domain': cookie_domain, 'secure': True
        })

    try:
        wait = WebDriverWait(driver, 20)
        driver.get(url_recebimento)
        driver.execute_script('window.scrollBy(0, 567);')

        campo_cod = wait.until(EC.presence_of_element_located((By.ID, 'seqCategoriaMovimentacao')))
        campo_cod.clear()
        campo_cod.send_keys('2011')
        time.sleep(tempo_espera)

        WebDriverWait(driver, 5).until(
            EC.element_to_be_clickable((By.ID, 'btnBuscaMovimentacao'))
        ).click()
        time.sleep(tempo_espera)

        campo_obs = wait.until(EC.presence_of_element_located((By.ID, 'observacao')))
        observacao = f'RECEBIDO Ofício ref: {assunto} em {data_resposta} - {hora_resposta} hs, por: {destinatario}'
        observacao = re.sub(r'\s+', ' ', observacao).strip()[:1500]
        campo_obs.clear()
        digitar(observacao, campo_obs)

        driver.execute_script('window.scrollBy(0, 567);')

        wait.until(EC.element_to_be_clickable((By.ID, 'Concluir'))).click()
        time.sleep(tempo_espera)

        alert = WebDriverWait(driver, 10).until(EC.alert_is_present())
        print(f'Alerta: {alert.text}')
        alert.accept()

        print('✅ Juntada realizada com sucesso!')
        time.sleep(tempo_espera)
        driver.quit()
        return True

    except Exception as e:
        print(f'❌ Erro na juntada: {e}')
        driver.quit()
        return False

In [29]:
# ============================================================
# PROCESSAMENTO DOS E-MAILS
# ============================================================

def processar_encaminhamento(msg, dados, msg_ids_registrados):
    """
    Encaminha e-mail se:
    - Tiver PDF anexado (fora da lista negra)
    - Ainda não tiver sido encaminhado (message_id não está no CSV)
    """
    msg_id = get_message_id(msg)  # ✅ Permanente, com fallbacks

    if not msg_id:
        print(f"⚠️ E-mail sem nenhum ID identificável, ignorado: {msg.subject}")
        return False

    # ✅ Checagem principal: já foi encaminhado?
    if msg_id in msg_ids_registrados:
        print(f"⏭️  Já encaminhado (msg_id no CSV): {msg.subject}")
        return False

    if not tem_pdf(msg):
        return False

    sucesso = encaminhar_email_completo(msg)
    if sucesso:
        registrar_encaminhamento(msg, dados, msg_ids_registrados, path_csv)

    return sucesso


def processar_resposta_projudi(msg, assuntos_map):
    """
    Registra resposta no Projudi se o assunto bater com um ofício enviado
    e ainda não tiver sido cumprido.
    """
    print("\n📩 NOVO E-MAIL =====================")
    print(f"Assunto original: {msg.subject}")

    assunto_msg = normalizar_assunto(msg.subject or '')
    print(f"Assunto normalizado: {assunto_msg}")
    print(f"Existe no assuntos_map? {assunto_msg in assuntos_map}")
    
    if assunto_msg not in assuntos_map:
        return False

    row = assuntos_map[assunto_msg]

    if row.get('registrar_envio') in ('recebido', 'cumprido', 'informado', 'devolvido', 'juntado'):
        print(f"⏭️  Resposta já registrada: {msg.subject}")
        return False

    texto = msg.text or (msg.html and BeautifulSoup(msg.html, 'html.parser').get_text()) or ''
    texto = limpar_cid(texto)
    texto = re.sub(r'\s+', ' ', texto).strip()[:100]

    data = msg.date.strftime('%d/%m/%Y')
    hora = msg.date.strftime('%H:%M:%S')

    assunto_juntada = f"{row['assunto'].split('Referente')[0].strip()} - {texto}"

    juntada_resposta_projudi(
        data_resposta    = data,
        hora_resposta    = hora,
        link_base        = link_base,
        destinatario     = msg.from_,
        url_recebimento  = row['url_recebimento'],
        assunto          = assunto_juntada,
        cookies          = cookies,
        cookie_domain    = cookie_domain,
        digitar          = digitar,
        tempo_espera     = tempo_espera
    )

    # Atualiza em memória
    row['status']          = 'recebido'
    row['resposta']        = texto
    row['registrar_envio'] = 'cumprido'
    return True

In [30]:
# ============================================================
# LOOP PRINCIPAL
# ============================================================

assuntos_map = montar_assuntos_map(dados)

encaminhados = 0
respondidos  = 0

with MailBox(IMAP_SERVER).login(usuario, senha_app, initial_folder='INBOX') as mailbox:
    mensagens = list(mailbox.fetch(criteria=A(all=True), reverse=True, limit=NUM_MAX_EMAILS))
    print(f"📬 {len(mensagens)} e-mails encontrados na caixa de entrada.")

    for msg in mensagens:
        if processar_encaminhamento(msg, dados, msg_ids_registrados):
            encaminhados += 1
        if processar_resposta_projudi(msg, assuntos_map):
            respondidos += 1

# Salva CSV atualizado ao final
salvar_csv(path_csv, dados)

print(f"\n📊 Resumo:")
print(f"   ✉️  Encaminhados nesta execução : {encaminhados}")
print(f"   📋 Respostas registradas        : {respondidos}")
print(f"   💾 Total no CSV                 : {len(dados)}")

📬 100 e-mails encontrados na caixa de entrada.
⏭️  Já encaminhado (msg_id no CSV): resp ao ofício nº 006/2026; 2ª VSJ - referente proc n° 0003353-71.2025.8.05.0191 

📩 NOVO E-MAIL =====================
Assunto original: resp ao ofício nº 006/2026; 2ª VSJ - referente proc n° 0003353-71.2025.8.05.0191 
Assunto normalizado: resp ao ofício nº 006/2026; 2ª vsj - referente proc n° 0003353-71.2025.8.05.0191
Existe no assuntos_map? True
⏭️  Resposta já registrada: resp ao ofício nº 006/2026; 2ª VSJ - referente proc n° 0003353-71.2025.8.05.0191 
⏭️  Já encaminhado (msg_id no CSV): Apresentação de Policial Militar

📩 NOVO E-MAIL =====================
Assunto original: Apresentação de Policial Militar
Assunto normalizado: apresentação de policial militar
Existe no assuntos_map? True
⏭️  Resposta já registrada: Apresentação de Policial Militar
⏭️  Já encaminhado (msg_id no CSV): Apresentação de Policial Militar

📩 NOVO E-MAIL =====================
Assunto original: Apresentação de Policial Militar